# 05 · Vanilla RNN 核心实现

目标：读懂单步前向、序列展开、BPTT、PyTorch Cell 和梯度传播。正式 Python 模块是唯一权威实现；本 Notebook 会动态提取并验证它。

In [ ]:
import inspect, os
from pathlib import Path
if not (Path.cwd() / 'common').exists() and (Path.cwd().parent / 'common').exists():
    os.chdir(Path.cwd().parent)
import matplotlib.pyplot as plt
import numpy as np
import torch
from common.gradcheck import check_gradient, numerical_gradient
from vanilla_rnn.numpy_impl import init_parameters, step_forward, sequence_forward, sequence_backward
from vanilla_rnn.torch_impl import ManualRNNCell, ManualRecurrentLayer
print('NumPy', np.__version__, '| PyTorch', torch.__version__)

## 1. 单步前向

$$a_t = x_t W_{ih}^T + b_{ih} + h_{t-1} W_{hh}^T + b_{hh}; h_t = tanh(a_t)$$

| 张量 | 形状 |
|---|---|
| $x_t$ | `[B,D]` |
| $h_{t-1}$ | `[B,H]` |
| $W_{ih}$ | `[H,D]` |
| $W_{hh}$ | `[H,H]` |
| $h_t$ | `[B,H]` |

In [ ]:
def notebook_rnn_step(x_t, h_prev, params):
    a_t = (x_t @ params['weight_ih'].T + params['bias_ih']
           + h_prev @ params['weight_hh'].T + params['bias_hh'])
    return np.tanh(a_t)

rng = np.random.default_rng(42)
example_params = init_parameters(3, 4, rng)
x_t, h_prev = rng.normal(size=(2, 3)), rng.normal(size=(2, 4))
reference_h, cache = step_forward(x_t, h_prev, example_params)
np.testing.assert_allclose(notebook_rnn_step(x_t, h_prev, example_params), reference_h)
print('单步输出形状:', reference_h.shape, '| cache:', sorted(cache))

## 2. 序列展开与 BPTT

前向从左到右复用同一组参数；反向从右到左。关键关系是：

$$dh_t = doutputs_t + dh_{future}; da_t = dh_t * (1-h_t^2)$$

参数在时间上共享，所以每个时间步的参数梯度必须累加。

In [ ]:
print(inspect.getsource(sequence_forward))
print(inspect.getsource(sequence_backward))

In [ ]:
rng = np.random.default_rng(7)
x = rng.normal(size=(2, 3, 2)).astype(np.float64)
h0 = rng.normal(size=(2, 3)).astype(np.float64)
params = init_parameters(2, 3, rng)
outputs, _, caches = sequence_forward(x, h0, params)
doutputs = rng.normal(size=outputs.shape)
dx, _, _ = sequence_backward(doutputs, caches)
def objective():
    current, _, _ = sequence_forward(x, h0, params)
    return float(np.sum(current * doutputs))
result = check_gradient('rnn-dx', dx, numerical_gradient(objective, x))
print(result)
assert result.passed

## 3. PyTorch Cell 与官方实现对齐

PyTorch 版本只省略手写反向；时间循环和单步公式不变。

In [ ]:
print(inspect.getsource(ManualRNNCell.forward))
manual = ManualRNNCell(3, 4).double()
official = torch.nn.RNNCell(3, 4).double()
official.load_state_dict(manual.state_dict())
x_t = torch.randn(2, 3, dtype=torch.float64)
h_prev = torch.randn(2, 4, dtype=torch.float64)
torch.testing.assert_close(manual(x_t, h_prev), official(x_t, h_prev))
print('官方 Cell 前向对齐通过')

In [ ]:
torch.manual_seed(42)
layer = ManualRecurrentLayer(3, 8).double()
sequence = torch.randn(1, 40, 3, dtype=torch.float64, requires_grad=True)
hidden, _ = layer(sequence)
hidden[:, -1].square().sum().backward()
gradient_norm = sequence.grad.norm(dim=(0, 2)).detach().numpy()
plt.semilogy(range(len(gradient_norm)), gradient_norm + 1e-20)
plt.xlabel('time index'); plt.ylabel('||dL/dx_t||'); plt.title('Vanilla RNN gradient flow');
plt.show()

## 4. 回忆区

关闭上面的答案后填写。默认不执行自测；完成函数后将开关改为 `True`。

In [ ]:
RUN_RECALL_TESTS = False
def recall_rnn_step(x_t, h_prev, params):
    # TODO: 只凭记忆完成单步公式
    pass
if RUN_RECALL_TESTS:
    np.testing.assert_allclose(recall_rnn_step(x_t.numpy(), h_prev.numpy(), example_params),
                               notebook_rnn_step(x_t.numpy(), h_prev.numpy(), example_params))
    print('回忆测试通过')

### 复盘问题

1. `dh_total` 为什么有两个来源？
2. 参数梯度为什么沿 time 累加？
3. `dh_next` 在逆序循环中代表什么？
4. 梯度裁剪能否解决长期记忆能力不足？